# 06 FID Evaluation

Compute FID from trained checkpoints.
`launch_eval_fid` generates samples in parallel on all GPUs and computes FID in the main process.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_eval_fid

cfg = load_cfg("configs/cifar10.yaml")
checkpoint_path = "outputs/checkpoints/cifar10_full_sdd_last.pt"

launch_eval_fid(
    cfg,
    checkpoint_path=checkpoint_path,
    num_samples=2048,      # Total number of fake samples
    num_processes=None,    # GPU auto-detection
)


In [ ]:
# Compare baseline vs SDD
from src.experiments import load_cfg, deep_update, launch_eval_fid

cfg_b = load_cfg("configs/cifar10.yaml")
cfg_b = deep_update(cfg_b, {
    "sdd": {"enabled": False, "lambda_distill": 0.0,
            "use_centering": False, "use_sharpening": False,
            "use_gating": False, "use_projection_head": False},
})

print("\n▶ Baseline FID:")
launch_eval_fid(cfg_b, "outputs/checkpoints/cifar10_baseline_mse_only_last.pt",
                num_samples=2048, num_processes=None)

print("\n▶ Full SDD FID:")
launch_eval_fid(load_cfg("configs/cifar10.yaml"),
                "outputs/checkpoints/cifar10_full_sdd_last.pt",
                num_samples=2048, num_processes=None)
